# 📗 Microsoft GraphRAG 색인과 네 가지 검색

**Microsoft GraphRAG**는 Microsoft Research가 2024년 4월 [논문](https://arxiv.org/abs/2404.16130)으로 발표하고, 같은 해 7월 `graphrag` 패키지로 공개한 GraphRAG입니다. 두 단계로 동작합니다.

1. **색인(질문 전에 미리):** LLM이 문서에서 개체와 관계를 뽑아 그래프를 만들고, 서로 많이 연결된 개체 묶음(커뮤니티)마다 요약문인 커뮤니티 보고서를 써 둡니다.
2. **검색(질문 때):** 질문에 맞춰 원문, 개체·관계, 커뮤니티 보고서 가운데 필요한 것을 골라 답변 모델에 넘깁니다.

이 교안은 `graphrag` 3.1.0으로 곡성·구례 관광지 안내문 32편을 색인하고, Basic·Local·Global·DRIFT 네 검색을 각자 잘 맞는 질문으로 실행해 **답변 모델에 어떤 근거가 넘어갔는지** 확인합니다.

**Neo4j로 직접 만든 GraphRAG 대신 왜 쓰나요?** 문서가 많고 형식이 제각각이면 스키마 설계가 가장 큰 일이고, 스키마에 없는 내용은 찾지 못합니다. 두 라이브러리는 LLM이 문서에서 바로 그래프를 만들고, 그래프 DB 없이 폴더 안 파일로 저장합니다. 대신 관계 이름이 없어서, 관계가 정해진 정확한 조회·집계는 Neo4j와 Cypher가 맞습니다.

<img src="images/01_why_graphrag_libs.png" width="1050" alt="앞 단원의 Neo4j GraphRAG와 Microsoft GraphRAG·LightRAG 비교: Neo4j는 사람이 스키마를 설계하고, 관계를 노고단 CONNECTED_TO 성삼재처럼 정해진 종류로 저장하며, Neo4j 서버에 두고, 결제 서비스 담당 팀 같은 정확한 조회·집계에 맞습니다. 두 라이브러리는 LLM이 문서에서 그래프를 추출하고, 관계를 설명 문장으로 저장하며, 폴더 안 파일에 두고, 되풀이된 장애 원인처럼 여러 문서를 모아 답하는 질문에 맞습니다.">

본실습은 **관광 안내 챗봇**이 곡성·구례 관광지 안내문(한국관광공사 TourAPI, 공공누리 제1유형)으로 답하는 상황입니다. 함께 따라하기는 **누리페이**(수업용 가상 회사) 운영팀의 장애 지식베이스입니다.

**학습 목표**

1. **Microsoft GraphRAG와 LightRAG를 색인 때 만드는 것과 질문 때 찾는 것으로 구분합니다.**
2. **`api.build_index`로 개체·관계·커뮤니티 보고서를 만들고, 그 근거 문서를 찾습니다.**
3. **Basic·Local·Global·DRIFT 검색을 실행하고, 답변 모델에 넘긴 원문을 문서 ID로 찾습니다.**
4. **검색마다 정답 근거 문서를 넘겼는지 세어 결과 표 한 행으로 만들고, 문서 한 편을 추가하는 비용을 잽니다.**

## 1. Microsoft GraphRAG와 LightRAG의 차이

### 1-1. 색인 때 만드는 것과 질문 때 찾는 것

두 방식 모두 LLM이 문서에서 개체와 관계를 뽑아 그래프를 만듭니다. 차이는 **색인 때 무엇을 더 만들어 두는지**와 **질문 때 무엇을 찾아 답변 모델에 넘기는지**입니다.

<img src="images/01_variant_map.png" width="1050" alt="Microsoft GraphRAG와 LightRAG 비교: 둘 다 LLM이 문서에서 개체·관계를 뽑아 그래프를 만듭니다. Microsoft GraphRAG는 개체를 커뮤니티로 묶어 커뮤니티 보고서를 미리 써 두고, 질문 때 커뮤니티 보고서와 개체·관계·원문을 찾습니다. LightRAG는 커뮤니티 보고서 없이 관계마다 키워드를 붙여 두고, 질문 때 키워드로 개체·관계를 벡터 검색합니다.">

| | Microsoft GraphRAG (교안 01) | LightRAG (교안 02) |
|---|---|---|
| 색인 때 만드는 것 | 개체·관계 그래프 + 커뮤니티 보고서(개체 묶음마다 LLM이 쓴 요약) | 개체·관계 그래프 + 관계 키워드(커뮤니티 보고서 없음) |
| 관계를 저장하는 방식 | 관계 이름(종류) 없이 설명 문장과 강도(`weight`) | 관계 이름(종류) 없이 설명 문장, 관계 키워드, 강도(`weight`) |
| 질문 때 찾는 것 | 검색 방식에 따라 원문, 개체·관계, 커뮤니티 보고서 | 질문에서 뽑은 키워드로 개체 설명과 관계 설명·키워드를 벡터 검색하고, 이어진 원문을 함께 |
| 색인 LLM 호출(관광 안내문 32편) | 147회 | 34회 |

**관계는 어떻게 찾나요?** 두 방식 모두 관계 이름이 없어 관계 종류로는 찾지 못합니다. 대신 임베딩을 쓰는데, 무엇을 임베딩해 찾는지가 다릅니다.

- **Microsoft GraphRAG(Local):** 질문과 비슷한 개체 설명을 벡터 검색해 개체를 찾고, 그 개체에 그래프로 이어진 관계를 함께 가져옵니다. 관계 자체를 벡터 검색하지는 않습니다.
- **LightRAG:** 관계마다 "관계 키워드 + 양 끝 개체 + 설명"을 임베딩해 두고, 질문의 주제 키워드와 비슷한 관계를 벡터 검색합니다.

**앞 단원 Neo4j 트리플과 무엇이 다른가요?** 같은 노고단·성삼재 관계를 세 방식으로 저장하면 이렇습니다. 아래 두 줄은 미리 만든 색인에 실제로 저장된 값입니다.

| | 저장된 관계 | 관계의 뜻이 있는 곳 |
|---|---|---|
| 앞 단원 Neo4j | `(:Peak {name: "노고단"})-[:CONNECTED_TO]->(:Pass {name: "성삼재"})` | 사람이 정한 관계 종류 `CONNECTED_TO`("이어져 있다") |
| Microsoft GraphRAG | 노고단 - 성삼재, 설명 "노고단은 성삼재를 기점으로 동쪽으로 연결되는 지리산의 주요 봉우리이다.", 강도 9 | 설명 문장 |
| LightRAG | 노고단 - 성삼재, 키워드 "등산로, 연결", 설명 "노고단은 성삼재를 기점으로 동쪽으로 연결되는 지리산 주요 봉우리이다." | 설명 문장과 키워드 |

- **Neo4j 트리플:** 관계 종류가 정해져 있어 `MATCH (:Pass {name: "성삼재"})-[:CONNECTED_TO]-(p:Peak)`처럼 "성삼재와 이어진 봉우리"만 정확히 거르고 셀 수 있습니다. 대신 스키마를 사람이 먼저 설계해야 합니다.
- **두 라이브러리:** 관계 이름(종류)을 뽑지 않고, 문서에 적힌 내용을 설명 문장으로 저장합니다. "이어져 있다"는 뜻은 문장 속 "연결되는"에만 있습니다. 관계 종류로 정확히 거르는 조회는 어렵지만, 스키마 없이 그래프를 만들고 설명 문장을 검색과 요약에 그대로 씁니다.
- **강도(`weight`):** 두 개체가 얼마나 강하게 이어졌는지 나타내는 수입니다. 같은 두 개체의 관계가 여러 청크에서 나오면 더해집니다.
  - Microsoft GraphRAG: LLM이 관계마다 매긴 점수입니다. 커뮤니티를 나눌 때 간선 가중치로 써서, 강하게 이어진 개체끼리 같은 커뮤니티로 묶이게 합니다.
  - LightRAG: LLM이 매기지 않고 관계가 나올 때마다 1씩 쌓입니다. `local` 검색에서 개체에 이어진 관계가 많을 때 어느 관계를 먼저 넘길지 정하는 데 씁니다.

**같은 개체를 두 방식은 이렇게 저장합니다.** 지리산·노고단·성삼재는 여러 안내문에 함께 나오는 개체입니다. 아래는 미리 만든 색인의 실제 값입니다.

- **Microsoft GraphRAG:** 셋을 한 커뮤니티로 묶고, 그 묶음 전체를 LLM이 요약한 커뮤니티 보고서를 색인 때 써 둡니다. 제목은 "지리산·성삼재·노고단 산악관광 및 마을 공동체"입니다. 그래서 "지리산 쪽 볼거리를 정리해 줘" 같은 넓은 질문에 미리 써 둔 이 요약을 바로 씁니다.
- **LightRAG:** 묶음과 요약 없이 관계 하나하나만 저장합니다. 노고단과 성삼재의 관계는 키워드 "등산로, 연결", 설명 "노고단은 성삼재를 기점으로 동쪽으로 연결되는 지리산 주요 봉우리이다."입니다. 넓은 질문에는 질문 때 관계와 원문 조각을 찾아 그 자리에서 모아 답합니다.

### 1-2. 질문 유형과 네 가지 검색

Microsoft GraphRAG의 네 검색은 잘 맞는 질문 유형과 답변 모델에 넘기는 근거가 다릅니다(예시의 S2·M1·G1은 정답 근거 문서를 미리 정해 둔 이 교안의 골드 질문입니다).

<img src="images/01_question_types.png" width="1050" alt="네 가지 검색과 잘 맞는 질문: 관광안내소 직원이 관광 안내 챗봇에 문의를 입력하는 네 장면입니다. Basic(단일 사실)은 대황강출렁다리 길이는? 질문에 비슷한 원문 조각을 찾고, Local(개체 주변)은 노고단 주변 볼거리는? 질문에 개체와 이웃·원문을 모읍니다. Global(전역)은 선비가 공부하던 곳 다 알려 줘 질문에 커뮤니티 보고서 전체를 요약하고, DRIFT(멀티홉)는 황전마을에서 노고단고개까지 얼마나 걸려? 질문에 후속 질문마다 Local 검색을 합니다.">

| 검색 | 잘 맞는 질문 | 답변 모델에 넘기는 것 | LLM 호출(질문 1개) |
|---|---|---|---|
| Basic | 단일 사실: 답이 문서 한 편에 있음(S2: 대황강출렁다리의 길이) | 질문과 가까운 원문 청크 5개 | 1회 |
| Local | 특정 개체 주변: 질문에 나온 개체를 중심으로 물음(예: 노고단 주변 볼거리) | 질문과 가까운 개체와, 그 개체에 이어진 관계 설명·원문 청크·커뮤니티 보고서 | 1회 |
| Global | 전역: 자료 전체에서 모아야 함(G1: 선비들이 공부하던 곳 모두) | 정한 레벨 이하의 커뮤니티 보고서 전부. 덩어리마다 요점을 뽑은 뒤(map) 하나로 합침(reduce) | 덩어리 수 + 1회(이 교안 약 15회) |
| DRIFT | 멀티홉: 문서 두 편 이상을 이어 읽어야 함(M1: 황전마을 옆 계곡의 본류를 따라 노고단고개까지 걸리는 시간) | 질문과 가까운 커뮤니티 보고서로 첫 답과 후속 질문을 만들고, 후속 질문마다 Local 검색한 결과를 합침 | 설정에 따라 다름(이 교안 6회) |

**Global은 왜 덩어리 수만큼 호출하나요?** Global은 임베딩으로 커뮤니티 보고서를 고르지 않고, 정한 레벨 이하의 커뮤니티 보고서를 **모두** 읽습니다. 모두 합치면 한 번에 넣기에 너무 길어서, 순서를 섞은 뒤 한 덩어리가 `max_context_tokens`(기본 12,000토큰)를 넘지 않게 나눕니다. 덩어리마다 LLM이 질문에 쓸 요점을 뽑고(map, 덩어리당 1회), 마지막에 요점을 모아 답을 씁니다(reduce, 1회). 그래서 커뮤니티 보고서가 많을수록 덩어리와 호출이 늘어납니다. 미리 만든 색인(레벨 0·1, 커뮤니티 보고서 73개)은 14덩어리라 15회입니다. 직접 색인하면 커뮤니티 보고서 수가 달라져 횟수도 달라집니다. 관련 있는 것만 골라 읽으려면 `dynamic_community_selection`을 켜는데, 이때도 임베딩이 아니라 LLM이 관련성 점수를 매겨 고릅니다.

**DRIFT 호출 수는 세 설정이 정합니다.**

| 설정 | 하는 일 | 기본값 | 이 교안 |
|---|---|---|---|
| `drift_k_followups` | 질문과 가까운 커뮤니티 보고서를 몇 개 고를지, 한 번에 실행할 후속 질문을 몇 개로 할지 | 20 | 3 |
| `primer_folds` | 고른 커뮤니티 보고서를 몇 덩어리로 나눠 첫 답을 만들지(덩어리당 LLM 1회) | 5 | 1 |
| `n_depth` | "후속 질문 -> Local 검색 -> 새 후속 질문"을 몇 번 반복할지 | 3 | 1 |

호출 수 = 질문 확장 1 + 첫 답 `primer_folds` + 후속 질문의 Local 검색 `n_depth` × `drift_k_followups` + 최종 답 1입니다. 이 교안은 1 + 1 + 1×3 + 1 = 6회, 기본값이면 최대 1 + 5 + 3×20 + 1 = 67회입니다. 후속 질문이 적게 나오면 이보다 덜 부릅니다.

**생각해 보기**: 전역 질문(G1)에 질문과 가장 가까운 문서 5편만 답변 모델에 넘기면 어떤 일이 생길까요?

<details><summary>정답 보기</summary>

5편 밖의 문서에만 있는 항목은 답변에서 빠집니다. 전역 질문에 답하려면 자료 전체를 미리 요약해 둔 결과가 필요합니다. 커뮤니티 보고서가 그 요약입니다.

</details>

### 1-3. Microsoft GraphRAG 색인 단계: 추출, 계층 커뮤니티, 커뮤니티 보고서

Microsoft GraphRAG의 색인은 네 단계입니다.

<img src="images/01_msgraphrag_index.png" width="1050" alt="Microsoft GraphRAG의 색인 과정: 문서를 청크로 나누고, LLM이 개체·관계를 추출해 그래프를 만들고, Leiden으로 큰 묶음과 작은 묶음의 계층 커뮤니티를 만든 뒤, 묶음마다 LLM이 커뮤니티 보고서를 요약해 씁니다.">

1. **추출:** 문서를 청크(일정 길이로 자른 조각)로 나누고, LLM이 청크마다 개체·관계를 뽑습니다. 예: 성삼재 안내문에서 `노고단`(장소), `지리산`(장소)과 둘의 관계 설명
2. **합치기:** 이름이 같은 개체는 하나로 합치고, 여러 설명은 LLM이 한 문단으로 요약합니다.
3. **계층 커뮤니티:** Leiden 알고리즘으로 많이 연결된 개체를 묶고, 묶음을 다시 작은 묶음으로 나눕니다. 레벨 0이 가장 큰 묶음입니다.
4. **커뮤니티 보고서:** 묶음마다 LLM이 커뮤니티 보고서(제목·요약·발견)를 씁니다.

**색인은 어디에 저장되나요?** Neo4j 같은 그래프 DB를 쓰지 않고, 프로젝트 폴더의 `output/`에 파일로 저장합니다. 이 교안에서는 `output/graphrag_본실습/output/`입니다.

- **그래프:** parquet 파일 여섯 개에 표 형태로 저장합니다. parquet는 pandas로 바로 읽는 열 단위 표 파일입니다.
- **임베딩:** 같은 폴더의 `lancedb/`에 저장합니다. LanceDB는 서버 없이 폴더 하나에 파일로 저장하는 벡터 DB라, 설치나 실행할 DB가 따로 없습니다.

| 파일 | 한 행이 담는 것 | 검색 때 쓰이는 방법 |
|---|---|---|
| `documents.parquet` | 입력 문서 한 편(제목 = 파일 이름) | 검색에는 쓰지 않음. 이 교안은 넘긴 청크의 문서 ID를 찾는 데 씀 |
| `text_units.parquet` | 청크 | Basic: 임베딩 검색 결과로 넘김 / Local: 찾은 개체가 나온 청크를 연결로 가져옴 |
| `entities.parquet` | 개체 이름·유형·설명 | Local의 입구. 질문과 비슷한 개체 설명을 임베딩으로 찾음(DRIFT는 후속 질문마다 하는 Local 검색에서 씀) |
| `relationships.parquet` | 두 개체의 관계 설명과 가중치 | 임베딩하지 않음. 찾은 개체에 이어진 관계만 연결로 가져옴 |
| `communities.parquet` | 개체 묶음(레벨, 포함한 청크) | 개체가 어느 커뮤니티에 속하는지 잇는 데 씀 |
| `community_reports.parquet` | 묶음마다 쓴 커뮤니티 보고서 | Global: 모두 읽음 / Local: 찾은 개체의 커뮤니티 것만 / DRIFT: 임베딩으로 가까운 것을 고름 |
| `lancedb/text_unit_text.lance` | 청크 임베딩 | Basic이 질문 임베딩과 비교 |
| `lancedb/entity_description.lance` | 개체 설명 임베딩 | Local(DRIFT의 후속 Local 포함)이 질문 임베딩과 비교 |
| `lancedb/community_full_content.lance` | 커뮤니티 보고서 본문 임베딩 | DRIFT 첫 단계가 가상 답변(질문으로 LLM이 쓴 글)의 임베딩과 비교 |

**추출한 것을 검색에서 어떻게 쓰나요?** 키워드(BM25)로 찾는 검색은 없고, 세 가지 방법만 씁니다.

1. **임베딩 유사도 검색(입구):** 질문(DRIFT는 질문으로 쓴 가상 답변)을 임베딩해 LanceDB에서 가까운 것을 찾습니다. 임베딩해 둔 것은 청크 원문, 개체 설명, 커뮤니티 보고서 본문 세 가지뿐입니다.
2. **연결 따라가기:** 찾은 개체에서 테이블의 id 연결(개체 -> 관계·청크·커뮤니티)을 따라 함께 넘길 것을 가져옵니다. 관계 설명은 임베딩하지 않아서 이 방법으로만 들어옵니다.
3. **통째로 읽기:** Global은 찾지 않고 정한 레벨 이하의 커뮤니티 보고서를 모두 LLM에 넣습니다.

## 2. 실습 준비

### 2-1. 환경·모델·경로

`graphrag` 3.1.0이 설치된 커널을 고릅니다(설치는 저장소 최상위에서 `uv sync`). OpenAI 키와 함께 따라하기에서 쓰는 Jev 키(`TYPESAFE_API_KEY`)는 `.env`에서 읽습니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# 노트북이 있는 단원 폴더의 data와 output을 사용합니다.
material_dir = Path(".").resolve()
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 노트북 폴더부터 위로 올라가며 처음 찾은 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 키는 저장소 최상위 .env에서 읽습니다(이미 읽은 키는 유지).
load_dotenv(material_dir.parent / ".env")


def read_json(name):
    """data 폴더 아래 name 경로의 JSON 파일을 읽어 list 또는 dict로 반환합니다.

    name: data 폴더 기준 상대 경로(예: "본실습/questions.json")
    """
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# 답변 모델(.env에 없으면 gpt-5.6-luna)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
# 답변 모델에 넘길 문서 수
TOP_K = 5

print("답변 모델:", model_name)

### 2-2. 곡성·구례 관광지 안내문과 골드 질문

`data/본실습/`의 안내문 33편 가운데 문서 추가 시연용 1편(`extra.json`)을 뺀 32편이 비교 세트입니다. 출처는 [자료 설명](data/본실습/README.md)에 있습니다.

In [ ]:
# 문서 추가 시연용 1편은 비교 세트에서 뺍니다.
extra_id = read_json("본실습/extra.json")["doc_id"]
# 한 줄에 문서 하나인 JSON Lines 파일이라 lines=True로 읽습니다.
docs_table = pd.read_json(data_dir / "본실습" / "docs.jsonl", lines=True)
docs_table = docs_table[docs_table["doc_id"] != extra_id]

print("비교 세트 문서:", len(docs_table), "편 | 군별 문서 수:", docs_table["county"].value_counts().to_dict())
display(docs_table.head())

골드 질문 12개를 읽고 유형마다 한 문항을 출력합니다. `gold_doc_ids`(정답 근거 문서)는 답하려면 읽어야 하는 문서 목록이고, 검색마다 이 가운데 몇 편을 넘겼는지 셉니다.

예를 들어 M1은 황전마을 안내문(옆 계곡의 본류가 화엄사계곡)과 화엄사계곡 안내문(노고단고개까지 걸리는 시간)을 모두 읽어야 답할 수 있어, 정답 근거가 두 편입니다.

In [ ]:
questions = read_json("본실습/questions.json")
# 질문 id로 찾도록 dict로 만듭니다.
questions_by_id = {question["id"]: question for question in questions}
question_table = pd.DataFrame(questions)

print("유형별 질문 수:", question_table["qtype"].value_counts().to_dict())
# 유형마다 한 문항(S2 단일 사실, M1 멀티홉, G1 전역)만 봅니다.
display(question_table[question_table["id"].isin(["S2", "M1", "G1"])][["id", "qtype", "question", "gold_doc_ids"]])

## 3. 색인 만들기

### 3-1. 설정 dict와 한국어 추출 프롬프트

GraphRAG의 동작은 설정 객체 `GraphRagConfig`가 정합니다. 바꿀 값만 dict에 적어 `GraphRagConfig(**settings)`로 만들고, 나머지는 기본값을 씁니다.

아래 셀은 프로젝트 폴더 `output/graphrag_본실습/`을 새로 만들고 문서 32편을 `input/`에 넣습니다. 이전 색인과 캐시도 지우므로, 색인을 처음부터 다시 하려면 이 셀부터 실행합니다.

In [ ]:
import shutil

project_dir = output_dir / "graphrag_본실습"
input_dir = project_dir / "input"
# 이전 색인·캐시와 입력을 지우고 문서를 새로 넣습니다.
# shutil.rmtree는 폴더를 안의 파일까지 지웁니다. ignore_errors=True면 폴더가 없어도 오류가 나지 않습니다.
shutil.rmtree(project_dir, ignore_errors=True)
input_dir.mkdir(parents=True)

# shutil.copy는 파일 하나를 폴더로 복사합니다. 파일 이름이 문서 제목이 되어 .txt를 떼면 문서 ID입니다.
for doc_id in docs_table["doc_id"]:
    shutil.copy(data_dir / "본실습" / "docs" / f"{doc_id}.txt", input_dir)

print("입력 문서:", len(list(input_dir.glob("*.txt"))), "편")

GraphRAG 패키지에는 LLM을 부르는 단계마다 Microsoft가 만든 기본 프롬프트가 들어 있습니다(예: `graphrag.prompts.index.extract_graph`의 `GRAPH_EXTRACTION_PROMPT`). 자료에 맞춘 프롬프트를 `graphrag prompt-tune` 명령으로 만들 수도 있습니다. 색인에 쓰는 프롬프트는 셋입니다.

- **개체·관계 추출:** 청크마다 개체·관계와 그 설명을 뽑습니다.
- **설명 요약:** 여러 문서에서 나온 같은 개체의 설명을 한 문단으로 합칩니다.
- **커뮤니티 보고서:** 묶음마다 제목·요약·주요 발견을 씁니다.

**왜 프롬프트를 고치나요?** GraphRAG에는 출력 언어 설정이 없습니다. 추출 프롬프트가 "영어로 출력"하라고 지시해 「노고단」 같은 이름이 영어(Nogodan)로 바뀌고, 요약·커뮤니티 보고서 프롬프트는 예시가 영어라 영어가 섞일 수 있습니다. 아래 셀은 세 프롬프트에 한국어 지시만 넣어 파일로 저장합니다. 설정이 프롬프트를 파일 경로로만 받기 때문입니다.

In [ ]:
# graphrag 패키지에 든 기본 프롬프트 문자열 세 개를 불러옵니다.
from graphrag.prompts.index.community_report import COMMUNITY_REPORT_PROMPT
from graphrag.prompts.index.extract_graph import GRAPH_EXTRACTION_PROMPT
from graphrag.prompts.index.summarize_descriptions import SUMMARIZE_PROMPT

# 추출 프롬프트: "영어로 출력" 지시를 한국어 지시로 바꿉니다.
extract_prompt = GRAPH_EXTRACTION_PROMPT.replace(
    "Return output in English as",
    "Write all descriptions in Korean. Keep entity names exactly as written in the source text "
    "(do not translate or romanize). Return output as")

# 설명 요약·커뮤니티 보고서 프롬프트: 언어 지시가 없어 맨 앞에 한국어 지시를 붙입니다.
korean_rule = "Write all text values in Korean.\n\n"
prompt_texts = {"extract_graph": extract_prompt,
                "summarize_descriptions": korean_rule + SUMMARIZE_PROMPT,
                "community_report": korean_rule + COMMUNITY_REPORT_PROMPT}

# 설정은 프롬프트 파일 경로만 받으므로 파일로 저장합니다.
prompt_paths = {}

for name, text in prompt_texts.items():
    prompt_paths[name] = output_dir / f"{name}_ko.txt"
    prompt_paths[name].write_text(text, encoding="utf-8")

`make_settings`는 설정 dict를 만드는 함수로, 함께 따라하기에서도 다시 씁니다. 공정하게 비교하려고, 원문을 넘기는 검색은 모두 문서 약 5편(`TOP_K`) 분량의 원문을 넘기게 맞춥니다.

- **Basic:** 청크 수 `k`로 맞춥니다.
- **Local·DRIFT:** 컨텍스트 가운데 원문에 쓸 비율(원문 비율, `text_unit_prop`)로 맞춥니다.

In [ ]:
import tiktoken

# GraphRAG와 같은 토크나이저로 문서 토큰 수를 셉니다.
encoding = tiktoken.get_encoding("o200k_base")


def make_settings(project_dir, entity_types):
    """project_dir/input의 .txt 문서를 색인하고 검색할 GraphRAG 설정 dict를 반환합니다. GraphRagConfig(**설정 dict)로 설정 객체를 만듭니다.

    project_dir: 프로젝트 폴더. input 폴더에서 문서를 읽고, 색인 결과·캐시·로그도 이 폴더 아래에 저장합니다.
    entity_types: LLM이 개체에 붙일 유형 이름 목록(예: ["장소", "인물", "단체", "행사"])
    Local·DRIFT가 원문 청크에 쓰는 비율(text_unit_prop)은 input 문서의 평균 토큰 수로 계산해 문서 TOP_K편 분량에 맞춥니다.
    """
    # 원문 비율 = 문서 TOP_K편 분량 토큰(원문 테이블 머리글 몫 50 포함) / Local 컨텍스트 한도(기본값 12,000)
    input_files = sorted((project_dir / "input").glob("*.txt"))
    total_tokens = 0

    for path in input_files:
        total_tokens += len(encoding.encode(path.read_text(encoding="utf-8")))

    text_unit_prop = round((TOP_K * total_tokens / len(input_files) + 50) / 12000, 3)
    # 키가 담기므로 설정 dict와 config는 출력하지 않습니다.
    api_key = os.environ["OPENAI_API_KEY"]
    # 속도 제한에 걸리면 기다렸다가 다시 부릅니다.
    retry = {"type": "exponential_backoff"}

    # 경로는 모두 문자열로 넣습니다.
    return {
        # 추출·요약·답변에 쓰는 채팅 모델
        "completion_models": {
            "default_completion_model": {"model_provider": "openai", "model": model_name,
                                         "api_key": api_key, "retry": retry},
        },
        # 임베딩 모델. 벡터를 768차원으로 줄입니다.
        "embedding_models": {
            "default_embedding_model": {"model_provider": "openai", "model": "text-embedding-3-large",
                                        "api_key": api_key, "retry": retry, "call_args": {"dimensions": 768}},
        },
        # 입력·출력·갱신·캐시·로그를 모두 프로젝트 폴더 아래에 둡니다(기본값은 노트북 폴더 기준).
        "input_storage": {"base_dir": str(project_dir / "input")},
        "output_storage": {"base_dir": str(project_dir / "output")},
        # 갱신 색인 때마다 이전 parquet 파일(previous)과 새 문서의 parquet 파일(delta)을 남기는 폴더
        "update_output_storage": {"base_dir": str(project_dir / "update_output")},
        # LLM 응답 캐시: 같은 요청을 다시 보내면 저장된 응답을 씁니다.
        "cache": {"storage": {"base_dir": str(project_dir / "cache")}},
        # 색인 단계별 실행 로그
        "reporting":{"base_dir": str(project_dir / "logs")},
        # 청크·개체 설명·커뮤니티 보고서 임베딩을 저장하는 벡터 저장소(기본값 LanceDB, 차원은 임베딩과 같은 768)
        "vector_store": {"db_uri": str(project_dir / "output" / "lancedb"), "vector_size": 768},
        # 개체·관계 추출: 개체 유형, 한국어 프롬프트, 빠진 개체를 다시 묻는 추가 추출은 끕니다.
        "extract_graph": {"entity_types": entity_types, "prompt": str(prompt_paths["extract_graph"]),
                          "max_gleanings": 0},
        # 같은 개체의 여러 설명을 한 문단으로 합치는 한국어 프롬프트
        "summarize_descriptions": {"prompt": str(prompt_paths["summarize_descriptions"])},
        # 커뮤니티 보고서를 쓰는 한국어 프롬프트
        "community_reports": {"graph_prompt": str(prompt_paths["community_report"])},
        # 떨어져 있는 개체 묶음도 커뮤니티로 나눕니다(기본값은 가장 큰 연결 묶음만).
        "cluster_graph": {"use_lcc": False},
        # Basic: 질문과 가까운 청크 TOP_K개를 넘깁니다.
        "basic_search": {"k": TOP_K},
        # Local: 컨텍스트 가운데 원문 청크에 쓸 비율(문서 TOP_K편 분량)
        "local_search": {"text_unit_prop": text_unit_prop},
        # DRIFT: 수업용으로 줄인 탐색 값, Local과 같은 원문 비율, temperature 1(gpt-5 계열은 1만 받음)
        "drift_search": {"primer_folds": 1, "drift_k_followups": 3, "n_depth": 1,
                         "local_search_text_unit_prop": text_unit_prop,
                         "local_search_temperature": 1, "reduce_temperature": 1},
    }

관광 안내문용 개체 유형으로 설정 객체를 만들고, dict 값이 제대로 들어갔는지 봅니다. 원문 비율 × 컨텍스트 한도가 Local이 원문에 쓰는 토큰 수입니다. 이 값이 문서 약 5편 분량이면 Basic(청크 5개)과 같은 양을 넘깁니다.

In [ ]:
from graphrag.config.models.graph_rag_config import GraphRagConfig

# 관광 안내문에 나오는 대상으로 개체 유형을 고릅니다.
settings = make_settings(project_dir, ["장소", "인물", "단체", "행사"])
# dict를 검사하고, 적지 않은 값은 기본값으로 채운 설정 객체를 만듭니다.
config = GraphRagConfig(**settings)

print("개체 유형:", config.extract_graph.entity_types, "| Basic 청크 수:", config.basic_search.k)

# Local이 원문 청크에 쓰는 토큰 = 원문 비율 x 컨텍스트 한도(기본값 12,000)
text_tokens = round(config.local_search.text_unit_prop * config.local_search.max_context_tokens)

print("원문 비율:", config.local_search.text_unit_prop, "| 원문에 쓰는 토큰:", text_tokens, "/", config.local_search.max_context_tokens)

### 3-2. `build_index` 실행과 비용 기록

`api.build_index`는 청크 나누기, 개체·관계 추출, 커뮤니티 나누기, 커뮤니티 보고서 쓰기, 임베딩을 차례로 실행합니다.

먼저 LLM 호출 수를 세는 준비를 합니다. GraphRAG는 litellm(여러 회사의 모델을 같은 방식으로 부르는 라이브러리)으로 모델을 부릅니다. litellm의 성공 콜백(호출이 성공할 때마다 불리는 함수)에 `record_chat_call`을 등록해, 호출 수와 토큰을 `usage`에 더합니다.

In [ ]:
import litellm

# 채팅 호출 수와 토큰을 더해 갈 dict
usage = {"llm_calls": 0, "input_tokens": 0, "output_tokens": 0}


async def record_chat_call(kwargs, response, start_time, end_time):
    """채팅 호출 하나가 성공하면 usage의 llm_calls를 1 늘리고 input_tokens·output_tokens에 이번 토큰 수를 더합니다.

    litellm.success_callback에 등록하면 litellm이 호출이 성공할 때마다 부릅니다. 임베딩 호출은 LLM 호출이 아니라 세지 않습니다.
    네 인자는 litellm이 넘기는 값이며, kwargs(호출 인자, call_type으로 호출 종류 확인)와 response(응답, response.usage에 토큰 수)만 씁니다.
    """
    # kwargs["call_type"]이 호출 종류입니다. GraphRAG는 채팅을 비동기 acompletion으로 부르고, 임베딩(aembedding)은 세지 않습니다.
    if kwargs["call_type"] == "acompletion":
        usage["llm_calls"] += 1
        # response.usage에 이번 호출의 입력(prompt)·출력(completion) 토큰 수가 있습니다.
        usage["input_tokens"] += response.usage.prompt_tokens
        usage["output_tokens"] += response.usage.completion_tokens


# 호출이 성공할 때마다 litellm이 이 함수를 부르도록 등록합니다.
litellm.success_callback = [record_chat_call]

색인을 실행하고 단계별 결과와 LLM 호출 수·토큰을 출력합니다. 호출 수에는 추출(청크마다), 설명 요약, 커뮤니티 보고서 작성(묶음마다)이 모두 들어갑니다. `api` 함수는 비동기 함수라 `await`를 붙여 부릅니다.

`method`는 색인 방식입니다.

- `"standard"`(기본값, 이 교안): 개체·관계 추출, 설명 요약, 커뮤니티 보고서를 모두 LLM이 합니다.
- `"fast"`: 개체는 LLM 대신 명사구 추출(NLP)로 뽑고, 함께 나온 명사구끼리 잇습니다. LLM은 원문 청크를 읽고 커뮤니티 보고서만 씁니다. 싸고 빠르지만 관계 설명이 없어 그래프가 거칩니다.

색인은 몇 분 걸리고 LLM을 약 150회 부릅니다. 기다리기 어렵다면 아래 색인 셀 대신 그다음 셀을 실행해, 강사가 같은 설정으로 미리 만든 색인(`data/본실습/prebuilt/`)을 복사합니다. 두 셀 가운데 하나만 실행합니다.

In [ ]:
import asyncio
import time

import graphrag.api as api

# 호출 수·토큰을 0으로 되돌리고 색인 시간을 잽니다.
usage.update(llm_calls=0, input_tokens=0, output_tokens=0)
start = time.time()
# 문서 읽기 -> 청크 나누기 -> 개체·관계 추출 -> 커뮤니티 나누기 -> 커뮤니티 보고서 작성 -> 임베딩을 차례로 실행합니다.
# method="standard"는 추출·요약·커뮤니티 보고서를 모두 LLM이 하는 기본 색인 방식입니다.
index_results = await api.build_index(config=config, method="standard")
index_seconds = time.time() - start
# litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
await asyncio.sleep(1)

# 단계(workflow)마다 성공과 실패를 출력합니다.
for result in index_results:
    if result.error is None:
        print("완료:", result.workflow)
    else:
        print("실패:", result.workflow, repr(result.error))

# 갱신 비용과 비교하려고 남겨 둡니다.
index_usage = dict(usage, seconds=round(index_seconds, 1))

print("색인 시간:", round(index_seconds), "초 | LLM 호출:", usage["llm_calls"], "회 | 입력 토큰:", usage["input_tokens"], "| 출력 토큰:", usage["output_tokens"])

In [ ]:
# 위 색인 셀 대신 실행합니다. 강사가 같은 설정으로 만든 색인을 project_dir/output에 복사합니다.
# shutil.copytree는 폴더를 하위 파일까지 복사합니다. dirs_exist_ok=True면 이미 있는 폴더에 덮어씁니다.
shutil.copytree(data_dir / "본실습" / "prebuilt" / "graphrag", project_dir / "output", dirs_exist_ok=True)
# 갱신 비용과 비교하려고 강사 색인의 호출 수·토큰·시간을 읽어 둡니다.
index_usage = read_json("본실습/prebuilt/index_usage.json")["graphrag"]

print("복사한 색인 파일:", sorted(path.name for path in (project_dir / "output").iterdir()))
print("강사 색인 시간:", round(index_usage["seconds"]), "초 | LLM 호출:", index_usage["llm_calls"], "회 | 입력 토큰:", index_usage["input_tokens"], "| 출력 토큰:", index_usage["output_tokens"])

### 3-3. 개체·관계·커뮤니티 테이블과 커뮤니티 보고서 읽기

`read_tables`는 색인이 만든 parquet 파일 여섯 개를 테이블(DataFrame)로 읽습니다.

In [ ]:
def read_tables(project_dir):
    """project_dir/output에 저장된 GraphRAG 색인 결과 parquet 파일 여섯 개를 읽어 {테이블 이름: DataFrame} dict로 반환합니다.

    테이블 이름: documents, text_units, entities, relationships, communities, community_reports
    """
    tables = {}

    # 색인 결과는 project_dir/output에 '테이블 이름.parquet' 파일로 저장됩니다.
    for name in ["documents", "text_units", "entities", "relationships", "communities", "community_reports"]:
        tables[name] = pd.read_parquet(project_dir / "output" / f"{name}.parquet")

    return tables


tables = read_tables(project_dir)

# 테이블마다 행 수와 열 이름을 봅니다.
for name, table in tables.items():
    print(name, len(table), "행 | 열:", list(table.columns))

LLM이 개체마다 붙인 유형(`type`)별 개수를 셉니다.

In [ ]:
# LLM이 개체마다 붙인 유형(type)별 개수
entity_type_counts = tables["entities"]["type"].value_counts()

display(entity_type_counts)

`relationships` 테이블에는 `LOCATED_IN`("~에 있다") 같은 관계 이름 열이 없습니다. 추출 프롬프트가 관계 하나를 (시작 개체, 끝 개체, 설명, 강도) 네 값으로만 받기 때문입니다. 관계의 뜻은 `description` 문장에, LLM이 매긴 강도는 `weight`에 있습니다. 아래 셀은 강도가 큰 관계 5개를 봅니다.

In [ ]:
relationships = tables["relationships"]

print("관계 수:", len(relationships))
# 강도(weight)가 큰 관계 5개
display(relationships.sort_values("weight", ascending=False)[["source", "target", "description", "weight"]].head())

개체 이름은 원문 표기를 따라서, 같은 장소도 「노고단」과 「지리산 노고단」, 「노고단 고개」와 「노고단고개」처럼 다른 개체로 저장될 수 있습니다. 그래서 아래 셀은 이름에 노고단이 들어 있는지(`str.contains`)로 관계를 고릅니다.

In [ ]:
# '지리산 노고단'처럼 달리 적힌 이름도 고르도록 포함 여부로 고릅니다.
mentions_nogodan = relationships["source"].str.contains("노고단") | relationships["target"].str.contains("노고단")
nogodan_relations = relationships[mentions_nogodan][["source", "target", "description", "weight"]]

display(nogodan_relations)

앞 단원의 스키마 그래프라면 `(:Peak {name: "노고단"})-[:LOCATED_IN]->(:Mountain {name: "지리산"})`처럼 관계 종류로 노고단이 어느 산에 있는지 바로 조회합니다. GraphRAG 그래프는 그런 조회가 어려운 대신, 커뮤니티 보고서로 넓은 질문을 요약하는 데 강합니다.

<img src="images/01_schema_vs_community.png" width="1050" alt="스키마 그래프와 커뮤니티 그래프: 왼쪽은 사람이 노드·관계 종류를 먼저 정한 그래프로 Cypher로 정확히 조회하며 정확한 조건 질문에 강합니다. 오른쪽은 문서에서 LLM이 개체·관계를 추출하고 커뮤니티 보고서를 붙인 그래프로 넓은 질문 요약에 강합니다.">

이제 커뮤니티 보고서를 봅니다. 먼저 레벨별 개수를 셉니다.

In [ ]:
# 레벨별 커뮤니티 보고서 수. 레벨 0이 가장 큰 묶음입니다.
report_level_counts = tables["community_reports"]["level"].value_counts().sort_index()

display(report_level_counts)

커뮤니티 보고서 한 행의 주요 열입니다.

- `title`: 제목
- `summary`: 전체 내용을 몇 문장으로 요약한 글
- `findings`: 주요 발견 목록. 발견마다 다시 `summary`(한 줄 제목)와 `explanation`(근거를 들어 풀어 쓴 설명, 끝에 개체·관계 번호 인용)이 있어, `summary`라는 이름이 두 수준에 쓰입니다.
- `full_content`: 제목·요약·발견을 합친 본문
- `rank`: LLM이 매긴 중요도

`community_reports` 테이블에는 근거 문서 열이 없어, 커뮤니티 보고서 → 커뮤니티 → 청크 → 문서 순서로 테이블을 이어 찾습니다. 아래 셀은 레벨 0에서 개체가 가장 많은 커뮤니티 보고서 하나의 근거 문서 ID를 찾습니다.

In [ ]:
reports = tables["community_reports"]
# 레벨 0 커뮤니티 보고서 가운데 묶인 개체가 가장 많은 것 하나를 고릅니다.
report = reports[reports["level"] == 0].sort_values("size", ascending=False).iloc[0]

# 커뮤니티 보고서는 커뮤니티 번호(community)로 communities 테이블과 이어집니다.
communities = tables["communities"]
community_row = communities[communities["community"] == report["community"]].iloc[0]
# communities에는 청크 ID만 있어 text_units 테이블을 거쳐 문서를 찾습니다.
text_units = tables["text_units"]
unit_rows = text_units[text_units["id"].isin(community_row["text_unit_ids"])]
documents = tables["documents"]
doc_rows = documents[documents["id"].isin(unit_rows["document_id"])]
# title은 입력 파일 이름이라 .txt를 떼면 문서 ID입니다.
report_doc_ids = sorted(doc_rows["title"].str.removesuffix(".txt"))

print("커뮤니티 보고서:", report["title"], "| 레벨:", report["level"], "| 묶인 개체 수:", report["size"])
# report["summary"]는 커뮤니티 보고서 전체를 요약한 글입니다.
print("커뮤니티 보고서 요약:", report["summary"])
print("근거 문서:", report_doc_ids)

그 커뮤니티 보고서의 첫 발견을 근거 원문과 대조해, LLM이 원문에 없는 내용을 적지 않았는지 봅니다. 발견 끝의 `엔터티 (...)`·`관계 (...)` 번호는 `entities`·`relationships` 테이블의 `human_readable_id`입니다.

In [ ]:
# 커뮤니티 보고서의 첫 발견과 근거 원문 앞 300자를 대조합니다.
# findings의 발견 하나에는 summary(한 줄 제목)와 explanation(근거를 든 설명)이 있습니다.
first_finding = report["findings"][0]

print("발견 제목:", first_finding["summary"])
print("발견 설명:", first_finding["explanation"])

for doc_id in report_doc_ids:
    original = (data_dir / "본실습" / "docs" / f"{doc_id}.txt").read_text(encoding="utf-8")
    print("\n[" + doc_id + "]", original[:300])

## 4. 네 가지 검색 실행과 비교

만든 색인으로 네 검색을 실행합니다. 각 `api.<이름>_search` 함수는 답변과 함께 **답변 모델에 넘긴 컨텍스트**(테이블 이름별 DataFrame dict)를 반환합니다. 검색마다 잘 맞는 유형의 질문 하나로 실행하고, 정답 근거 문서를 넘겼는지 봅니다. 여러 질문의 결과를 모아 방식을 비교하는 일은 부록에서 합니다.

<img src="images/01_search_flows.png" width="1050" alt="네 가지 검색이 답변 모델에 넘기는 것: Basic은 질문으로 청크 벡터를 검색해 원문 청크 5개를 답변 모델에 넘깁니다. Local은 질문으로 개체 설명 벡터를 검색하고, 그 개체에 이어진 관계·원문·커뮤니티 보고서를 답변 모델에 넘깁니다. Global은 커뮤니티 보고서 전부를 덩어리로 나눠 덩어리마다 LLM이 요점을 뽑고(map), 요점을 모아 LLM이 답을 씁니다(reduce). DRIFT는 질문과 가까운 커뮤니티 보고서로 LLM이 첫 답을 만들고, 후속 질문마다 Local 검색을 한 뒤 모아서 최종 답을 씁니다.">

그림은 네 검색이 질문에서 답변 모델까지 거치는 길이고, 노란 말풍선이 LLM 호출입니다. 네 검색은 위에서 본 세 방법(임베딩 유사도 검색, 연결 따라가기, 통째로 읽기)을 다르게 조합합니다. 검색마다 자세한 단계는 각 검색 절에서 설명합니다.

아래 셀은 네 검색이 함께 쓰는 값을 정합니다.

- `community_level`: 쓸 커뮤니티 보고서 레벨의 상한입니다. 올릴수록 Global이 읽는 양과 비용이 늘어 1로 둡니다.
- `response_type`(답 형식): 검색 프롬프트의 `{response_type}` 자리에 들어가는 문장입니다. 프롬프트가 영어라 정하지 않으면 영어로 길게 답하기 쉽습니다. 그래서 전역 질문은 한국어 목록으로, 나머지는 한국어 한두 문장으로 받습니다.

In [ ]:
# Global이 읽을 커뮤니티 보고서 레벨 상한. 이 레벨 이하(레벨 0·1)를 모두 씁니다.
# 레벨을 올리면 더 작은 묶음의 커뮤니티 보고서까지 더해져 map 호출과 입력 토큰이 늘어납니다.
community_level = 1
# response_type으로 넘길 답 형식. 전역 질문은 목록으로, 나머지는 한두 문장으로 받습니다.
SHORT_ANSWER = "한국어로 한두 문장. 근거에 없으면 모른다고 답한다"
LIST_ANSWER = "한국어로, 해당하는 항목을 모두 나열한 목록"

DRIFT는 검색 한 번에 Local 검색을 여러 번 하지만, api 반환값에는 마지막 컨텍스트만 남습니다. 그래서 콜백으로 컨텍스트를 모두 모읍니다.

- `QueryCallbacks`: GraphRAG가 검색 도중 단계마다 부르는 함수를 모아 둔 클래스입니다.
- `ContextCollector`: 그 가운데 컨텍스트가 만들어질 때마다 불리는 `on_context`만 바꿔, 컨텍스트를 목록에 쌓습니다.

In [ ]:
from graphrag.callbacks.query_callbacks import QueryCallbacks


class ContextCollector(QueryCallbacks):
    """GraphRAG 검색에 넘기는 콜백입니다. 검색이 답변 모델에 넘길 컨텍스트가 만들어질 때마다 contexts 목록에 쌓습니다.

    contexts의 원소 하나는 {테이블 이름: DataFrame} dict입니다. 검색이 끝난 뒤 contexts를 읽으면 그 검색이 넘긴 컨텍스트를 모두 볼 수 있습니다.
    """

    def __init__(self):
        # 컨텍스트(테이블 이름 -> DataFrame dict)를 불린 순서대로 쌓습니다.
        self.contexts = []

    def on_context(self, context):
        """만들어진 컨텍스트({테이블 이름: DataFrame} dict)를 contexts 끝에 덧붙입니다."""
        # 컨텍스트가 만들어질 때마다 불립니다. DRIFT는 후속 Local 검색마다 불립니다.
        self.contexts.append(context)

`run_search`는 이름(`mode`)으로 네 검색 가운데 하나를 실행해 답, 모은 컨텍스트, 걸린 시간(초)을 돌려줍니다. 따라하기 색인에도 쓰도록 설정과 테이블을 인자로 받습니다. 네 `api.*_search` 함수의 주요 인자는 이렇습니다.

| 인자 | 뜻 | 쓰는 검색 |
|---|---|---|
| `config` | 모델·벡터 저장소와 검색 값(청크 수 `k`, 원문 비율 등)을 담은 설정 객체 | 모두 |
| `query` | 질문 문장 | 모두 |
| `text_units` | 원문 청크 테이블 | Basic, Local, DRIFT |
| `entities`, `relationships` | 개체·관계 테이블 | Local, DRIFT (Global은 `entities`만) |
| `communities`, `community_reports` | 커뮤니티와 커뮤니티 보고서 테이블 | Local, Global, DRIFT |
| `community_level` | 쓸 커뮤니티 보고서 레벨의 상한 | Local, Global, DRIFT |
| `covariates` | 주장(claim) 테이블. 이 교안은 만들지 않아 `None` | Local |
| `dynamic_community_selection` | 관련 커뮤니티 보고서만 골라 읽을지 | Global |
| `response_type` | 답 형식 문장 | 모두 |
| `callbacks` | 검색 도중 불릴 콜백 목록(여기서는 컨텍스트 수집) | 모두 |

In [ ]:
import asyncio
import time

import graphrag.api as api


async def run_search(mode, question, config, tables):
    """질문 하나에 GraphRAG 검색 하나(mode)를 실행해 (answer, contexts, seconds)를 반환합니다.

    mode: "basic", "local", "global", "drift" 가운데 하나
    question: 골드 질문 dict. qtype이 "global"이면 항목을 모두 나열하게, 나머지는 한두 문장으로 답하게 합니다.
    config: GraphRAG 설정 객체(GraphRagConfig)
    tables: read_tables가 반환한 테이블 dict
    반환값: answer는 답변 글, contexts는 검색이 답변 모델에 넘긴 컨텍스트 목록(ContextCollector가 모은 것), seconds는 검색에 걸린 시간(초)
    시작할 때 usage를 0으로 되돌리므로, 끝난 뒤 usage에는 이 검색 한 번의 사용량만 남습니다.
    """
    # 검색마다 새 콜백을 만들어 앞 검색의 컨텍스트가 섞이지 않게 합니다.
    collector = ContextCollector()

    # 전역 질문은 항목을 모두 나열하게, 나머지는 짧게 답하게 합니다.
    if question["qtype"] == "global":
        response_type = LIST_ANSWER
    else:
        response_type = SHORT_ANSWER

    # 이 검색의 사용량만 세도록 0으로 되돌립니다.
    usage.update(llm_calls=0, input_tokens=0, output_tokens=0)
    start = time.time()

    # 반환값은 (답, 마지막 컨텍스트)입니다. 컨텍스트는 콜백이 모은 목록을 쓰므로 버립니다.
    if mode == "basic":
        # Basic: text_units 테이블만 넘기고, 질문과 가까운 청크를 벡터 저장소에서 찾습니다.
        answer, _ = await api.basic_search(
            config=config, text_units=tables["text_units"],
            response_type=response_type, query=question["question"], callbacks=[collector])
    elif mode == "local":
        # Local: 개체·관계·커뮤니티·커뮤니티 보고서·청크 테이블을 모두 씁니다. 주장(covariates) 테이블은 만들지 않아 None입니다.
        answer, _ = await api.local_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], text_units=tables["text_units"],
            relationships=tables["relationships"], covariates=None, community_level=community_level,
            response_type=response_type, query=question["question"], callbacks=[collector])
    elif mode == "global":
        # Global: 커뮤니티·커뮤니티 보고서 테이블로 map-reduce를 합니다. 원문 청크 테이블은 넘기지 않습니다.
        answer, _ = await api.global_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], community_level=community_level,
            # False: 레벨 상한 이하 커뮤니티 보고서를 모두 읽습니다. True면 LLM이 관련 있는 것만 골라 읽습니다.
            dynamic_community_selection=False, response_type=response_type, query=question["question"],
            callbacks=[collector])
    elif mode == "drift":
        # DRIFT: 커뮤니티 보고서로 첫 답을 만들고 후속 질문마다 Local 검색을 해서 Local과 같은 테이블이 필요합니다.
        answer, _ = await api.drift_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], text_units=tables["text_units"],
            relationships=tables["relationships"], community_level=community_level,
            response_type=response_type, query=question["question"], callbacks=[collector])

    seconds = time.time() - start
    # litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
    await asyncio.sleep(1)
    return answer, collector.contexts, seconds

### 4-1. Basic: 청크 벡터 검색

Basic은 그래프를 쓰지 않고 질문과 가까운 원문 청크만 넘기는 벡터 검색입니다. 나머지 세 검색의 비교 기준이 됩니다.

**동작**

1. 질문을 임베딩합니다.
2. LanceDB `text_unit_text`(청크 임베딩)에서 질문과 가장 가까운 청크 `k`개(기본 10개, 이 교안 5개)를 찾습니다.
3. 찾은 청크 원문을 원문 테이블(`sources`) 하나로 답변 모델에 넘깁니다. LLM은 답을 쓸 때 1회 부릅니다.

**맞는 질문과 약점:** 답이 문서 한 편에 있는 단일 사실 질문에 맞습니다. 답이 여러 문서에 흩어져 있거나 질문과 표현이 다른 문서는 놓치기 쉽습니다.

아래 셀은 단일 사실 질문 S2로 실행합니다. 답 끝의 `[Data: Sources (번호)]`는 인용한 청크 번호이고, 문서 ID가 아닙니다.

In [ ]:
question = questions_by_id["S2"]

print("질문", question["id"] + ":", question["question"])

basic_answer, basic_contexts, basic_seconds = await run_search("basic", question, config, tables)

print("답:", basic_answer)
# Basic은 컨텍스트 하나에 원문 청크 테이블(sources)만 넘깁니다. id 열이 청크 번호입니다.
print("넘긴 테이블:", list(basic_contexts[0].keys()), "| sources의 청크 번호:", basic_contexts[0]["sources"]["id"].tolist())

원문 테이블(`sources`)은 청크를 번호로만 가리킵니다. `context_doc_ids`는 청크 번호를 문서 ID로 바꿉니다(`text_units` 테이블과 `documents` 테이블을 잇습니다). DRIFT처럼 컨텍스트가 여러 개일 수 있어 컨텍스트 목록을 인자로 받습니다. 아래 셀은 Basic이 넘긴 문서 ID를 S2의 정답 근거 문서와 나란히 출력합니다.

In [ ]:
def context_doc_ids(contexts, tables):
    """run_search가 모은 컨텍스트의 원문 테이블(sources)에 든 청크가 나온 문서 ID를 중복 없이 정렬한 list로 반환합니다.

    contexts: run_search가 반환한 컨텍스트 목록. 원소 하나가 {테이블 이름: DataFrame} dict이며, sources가 없는 컨텍스트(Global의 커뮤니티 보고서)는 건너뜁니다.
    tables: read_tables가 반환한 테이블 dict. text_units·documents 테이블로 청크 번호를 문서 ID로 바꿉니다.
    """
    # 청크 번호 -> 청크 행, 문서 내부 id -> 문서 행으로 바로 찾게 합니다.
    units = tables["text_units"].set_index("human_readable_id")
    documents = tables["documents"].set_index("id")

    doc_ids = set()

    for context in contexts:
        # 원문 테이블이 없는 컨텍스트(Global의 커뮤니티 보고서)는 건너뜁니다.
        if "sources" not in context:
            continue
        # sources의 id는 청크 번호를 문자열로 담아 int로 바꿔 찾습니다.
        for unit_id in context["sources"]["id"]:
            document_id = units.loc[int(unit_id), "document_id"]
            # title은 입력 파일 이름이라 .txt를 떼면 문서 ID입니다.
            doc_ids.add(documents.loc[document_id, "title"].removesuffix(".txt"))

    return sorted(doc_ids)


basic_ids = context_doc_ids(basic_contexts, tables)

# Basic이 넘긴 문서와 S2의 정답 근거 문서를 나란히 봅니다.
print("Basic이 넘긴 문서 ID:", basic_ids)
print("S2 정답 근거 문서 ID:", questions_by_id["S2"]["gold_doc_ids"])

`evidence_hit`는 넘긴 문서에 든 정답 근거 문서 수를 셉니다.

In [ ]:
def evidence_hit(given_ids, gold_ids):
    """정답 근거 문서 가운데 검색이 넘긴 문서가 몇 편인지 세어 int로 반환합니다.

    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록
    gold_ids: 질문의 정답 근거 문서 ID 목록(골드 질문의 gold_doc_ids)
    예: 정답 근거 3편 가운데 2편을 넘겼으면 2를 반환합니다.
    """
    hit = 0

    # 정답 근거 문서마다 넘긴 목록에 들어 있는지 봅니다.
    for gold_id in gold_ids:
        if gold_id in given_ids:
            hit += 1

    return hit


# Basic이 S2의 정답 근거 문서를 몇 편 넘겼는지 셉니다.
print("Basic이 넘긴 S2 정답 근거:", evidence_hit(basic_ids, questions_by_id["S2"]["gold_doc_ids"]), "편")

`make_result_row`는 검색 한 번의 결과를 표 한 행으로 만듭니다. Global처럼 원문을 넘기지 않으면 문서 열(`docs_given`·`gold_docs`·`hit_docs`·`all_hit`)을 빈칸으로 둡니다. `all_hit`는 정답 근거 문서를 모두 넘겼으면 1입니다. 아래 셀은 Basic의 S2 결과로 한 행을 만듭니다.

In [ ]:
def make_result_row(method, question, given_ids, usage, seconds):
    """검색 한 번의 결과를 비교용 결과 표의 한 행(dict)으로 만들어 반환합니다.

    method: 검색 방식 이름(예: "graphrag_local", "lightrag_mix")
    question: 골드 질문 dict(id, qtype, gold_doc_ids)
    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록. 원문을 넘기지 않는 GraphRAG Global은 None을 넘기고, 이때 문서 열은 빈칸(None)이 됩니다.
    usage: 이 검색의 LLM 사용량 dict(llm_calls 호출 수, input_tokens 입력 토큰, output_tokens 출력 토큰)
    seconds: 검색에 걸린 시간(초)
    문서 열: docs_given(넘긴 문서 수), gold_docs(정답 근거 수), hit_docs(정답 근거 가운데 넘긴 수), all_hit(모두 넘겼으면 1, 아니면 0)
    """
    # 사용량과 시간은 모든 방식에 채우고, 문서 열은 빈칸(None)으로 시작합니다.
    row = {"method": method, "question_id": question["id"], "qtype": question["qtype"],
           "docs_given": None, "gold_docs": None, "hit_docs": None, "all_hit": None,
           "llm_calls": usage["llm_calls"], "input_tokens": usage["input_tokens"],
           "output_tokens": usage["output_tokens"], "seconds": round(seconds, 1)}

    # 원문 문서를 넘긴 방식만 문서 열을 채웁니다.
    if given_ids is not None:
        gold_ids = question["gold_doc_ids"]
        hit = evidence_hit(given_ids, gold_ids)
        row["docs_given"] = len(given_ids)
        row["gold_docs"] = len(gold_ids)
        row["hit_docs"] = hit

        # 정답 근거를 모두 넘겼으면 1
        if hit == len(gold_ids):
            row["all_hit"] = 1
        else:
            row["all_hit"] = 0

    return row


# usage에는 바로 앞 Basic 검색의 사용량이 들어 있습니다.
basic_row = make_result_row("graphrag_basic", questions_by_id["S2"], basic_ids, usage, basic_seconds)

display(pd.DataFrame([basic_row]))

### 4-2. Local: 개체와 연결된 관계·커뮤니티 보고서·원문

Local은 질문과 가까운 개체를 출발점으로, 그래프에서 그 개체에 이어진 관계·커뮤니티 보고서·원문 청크를 모아 넘기는 검색입니다.

**동작**

1. 질문을 임베딩해 LanceDB `entity_description`(개체 설명 임베딩)에서 가까운 개체 10개(`top_k_entities` 기본값)를 찾습니다.
2. 찾은 개체에서 그래프를 따라 세 가지를 모읍니다. 관계는 벡터 검색하지 않고, 찾은 개체에 이어진 것만 가져옵니다.
   - 관계: 찾은 개체끼리의 관계를 먼저, 이어 바깥 개체로 가는 관계를 연결 수·순위가 높은 순으로
   - 커뮤니티 보고서: 찾은 개체가 많이 속한 커뮤니티 순으로
   - 원문 청크: 찾은 개체가 나온 청크(질문과 가까운 개체의 청크부터)
3. 컨텍스트 한도 12,000토큰을 몫으로 나눠 채웁니다. 커뮤니티 보고서 15%(`community_prop`), 원문 청크는 원문 비율(`text_unit_prop`, 문서 약 5편 분량), 나머지가 개체·관계입니다. 몫을 넘는 행은 넣지 않습니다.
4. 개체·관계·커뮤니티 보고서·원문 네 테이블을 한 컨텍스트로 답변 모델에 넘깁니다. LLM은 1회입니다.

**맞는 질문과 약점:** 질문에 나온 특정 개체 주변을 묻는 질문에 맞습니다. 출발 개체를 잘못 고르면 엉뚱한 이웃이 넘어가고, 원문은 원문 비율만큼만 들어가 개체·관계 설명만으로 답하기도 합니다.

아래 셀은 멀티홉 질문 M1로 실행하고 넘긴 테이블별 행 수를 출력합니다. M1에 답하려면 황전마을 안내문(황전 계곡이 화엄사계곡의 지류)과 화엄사계곡 안내문(노고단고개까지 걸리는 시간)이 함께 넘어가야 합니다.

In [ ]:
question = questions_by_id["M1"]

print("질문", question["id"] + ":", question["question"])

# 걸린 시간은 쓰지 않아 _로 받습니다.
local_answer, local_contexts, _ = await run_search("local", question, config, tables)

print("답:", local_answer)

# 주장(claim) 추출을 끈 설정이라(covariates=None) 넘긴 테이블에 claims는 없습니다.
# Local 컨텍스트는 개체·관계·커뮤니티 보고서·원문 테이블을 한 dict에 담습니다.
for name, table in local_contexts[0].items():
    print("넘긴", name, "테이블:", len(table), "행")

Local이 넘긴 문서 ID를 M1의 정답 근거 문서와 견줍니다. Local은 다음 청크가 토큰 한도를 넘으면 멈추므로 5편보다 적게 넘길 수 있습니다.

In [ ]:
# 원문 테이블(sources)의 청크만 문서 ID로 바꿉니다. 개체·관계 설명은 문서 ID가 없습니다.
local_ids = context_doc_ids(local_contexts, tables)

print("Local이 넘긴 문서 ID:", local_ids)
print("M1 정답 근거 문서 ID:", questions_by_id["M1"]["gold_doc_ids"], "| 넘긴 정답 근거:", evidence_hit(local_ids, questions_by_id["M1"]["gold_doc_ids"]), "편")

### 4-3. Global: 커뮤니티 보고서 map-reduce

Global은 질문으로 검색하지 않고, 커뮤니티 보고서를 모두 나눠 읽어 요점을 모으는 map-reduce 검색입니다.

**동작**

1. 임베딩을 쓰지 않습니다. 레벨 상한(`community_level`) 이하의 커뮤니티 보고서를 모두 가져옵니다(이 교안 레벨 0·1, 미리 만든 색인 기준 73개).
2. 순서를 섞은 뒤 한 덩어리가 12,000토큰(`max_context_tokens`)을 넘지 않게 나눕니다(미리 만든 색인 기준 14덩어리).
3. map: 덩어리마다 LLM이 질문에 쓸 요점과 중요도 점수(0~100)를 냅니다. 덩어리당 1회이고, 여러 덩어리를 동시에 부릅니다.
4. reduce: 점수 0인 요점을 버리고 점수가 높은 순으로 모아 LLM이 최종 답을 씁니다(1회). 미리 만든 색인 기준 모두 15회입니다.

**맞는 질문과 약점:** 자료 전체에서 항목을 모아야 하는 전역 질문에 맞습니다. 원문을 넘기지 않아 커뮤니티 보고서에 없는 세부는 답에 담지 못하고, 커뮤니티 보고서가 많을수록 호출과 입력 토큰이 늘어납니다.

`dynamic_community_selection`은 읽을 커뮤니티 보고서를 고르는 방식입니다.

- `False`(이 교안): 위 동작대로 레벨 상한 이하를 모두 읽습니다.
- `True`: map 전에 LLM이 레벨 0 커뮤니티 보고서부터 하나씩 질문과 관련 있는지 점수를 매기고, 관련 있는 것의 하위 커뮤니티로 내려가며 고른 것만 map에 넣습니다. 이때도 임베딩은 쓰지 않습니다. 점수를 매기는 호출이 더 들지만 map에 넣는 양이 줄어, 운영에서 Global 비용을 줄일 때 먼저 켜 보는 옵션입니다.

아래 셀은 전역 질문 G1로 실행합니다. 답 끝의 `[Data: Reports (번호)]`는 인용한 커뮤니티 보고서 번호입니다. Global은 원문을 넘기지 않으므로, 정답 근거 문서의 관광지 이름을 출력해 답 목록에 빠짐없이 들었는지 견줍니다.

In [ ]:
from IPython.display import Markdown

question = questions_by_id["G1"]

print("질문", question["id"] + ":", question["question"])

global_answer, global_contexts, _ = await run_search("global", question, config, tables)

# reports 테이블은 map 단계에 나눠 넣은 커뮤니티 보고서 목록입니다.
print("map에 나눠 넣은 커뮤니티 보고서:", len(global_contexts[0]["reports"]), "개")
# 호출은 map 여러 번과 마지막 reduce 1번입니다.
print("LLM 호출:", usage["llm_calls"], "회 (map", usage["llm_calls"] - 1, "회 + reduce 1회) | 입력 토큰:", usage["input_tokens"])

# 원문 대신 정답 근거 문서의 관광지 이름을 답 목록과 견줍니다.
gold_titles = docs_table[docs_table["doc_id"].isin(question["gold_doc_ids"])]["title"].tolist()

print("G1 정답 근거 문서의 관광지:", gold_titles)
# 목록 답이라 마크다운으로 렌더링해 봅니다.
print("답:")
display(Markdown(global_answer))

### 4-4. DRIFT: 커뮤니티 보고서로 첫 답, 후속 질문마다 Local 검색

DRIFT는 질문과 가까운 커뮤니티 보고서 몇 개로 첫 답과 후속 질문을 만들고, 후속 질문마다 Local 검색을 해 답을 합치는 검색입니다. Global과 달리 커뮤니티 보고서를 전부 읽지 않습니다. 괄호 안의 횟수는 이 교안 설정(`primer_folds` 1, `drift_k_followups` 3, `n_depth` 1) 기준이고, 세 설정의 뜻은 위 DRIFT 설정 표에 있습니다.

**동작**

1. 질문 확장(LLM 1회): 커뮤니티 보고서 하나를 무작위로 골라 그 형식대로 가상 답변을 쓰게 하고, 질문 대신 이 가상 답변을 임베딩합니다(HyDE 방식).
2. LanceDB `community_full_content`(커뮤니티 보고서 본문 임베딩)와 비교해 가장 가까운 커뮤니티 보고서 `drift_k_followups`개(3개)를 고릅니다.
3. 첫 답(LLM `primer_folds`회, 1회): 고른 커뮤니티 보고서로 첫 답과 후속 질문을 JSON으로 받습니다.
4. 후속 질문 `drift_k_followups`개(3개)마다 Local 검색을 합니다(질문당 LLM 1회). 원문은 이 단계에서 들어가고, 각 Local 검색이 새 후속 질문도 냅니다. 이 단계를 `n_depth`번(1번) 반복합니다.
5. 첫 답과 Local 검색의 답을 모아 최종 답을 씁니다(reduce, LLM 1회). 이 교안은 모두 6회입니다.

**맞는 질문과 약점:** 문서 여러 편을 이어 읽어야 하는 멀티홉 질문에 맞습니다. 호출이 많고, 가상 답변의 틀과 실행할 후속 질문을 무작위로 골라 같은 질문도 실행마다 넘기는 문서가 달라질 수 있습니다.

첫 답이 정해진 JSON 형식에 맞지 않으면 에러가 나므로, 그때는 셀을 다시 실행합니다.

아래 셀은 M1(황전마을 옆 계곡의 본류를 따라 노고단고개까지 걸리는 시간)로 `api.drift_search`를 직접 불러, 반환값과 콜백이 모은 컨텍스트의 문서 ID를 비교합니다. 반환값에는 마지막 Local 검색의 컨텍스트만 남아, 반환값만으로 세면 넘긴 문서를 실제보다 적게 셀 수 있습니다.

In [ ]:
question = questions_by_id["M1"]

print("질문", question["id"] + ":", question["question"])

# api 반환값과 콜백이 모은 컨텍스트를 비교합니다.
collector = ContextCollector()

# 반환값을 보려고 run_search 대신 api.drift_search를 직접 부릅니다. 넘기는 테이블은 run_search와 같습니다.
drift_answer, drift_context = await api.drift_search(
    config=config, entities=tables["entities"], communities=tables["communities"],
    community_reports=tables["community_reports"], text_units=tables["text_units"],
    relationships=tables["relationships"], community_level=community_level,
    response_type=SHORT_ANSWER, query=question["question"], callbacks=[collector])

print("답:", drift_answer)
print("on_context가 불린 횟수:", len(collector.contexts))
# 반환값에는 마지막 Local 검색의 컨텍스트 하나만 들어 있습니다.
print("api가 돌려준 컨텍스트의 문서 ID:", context_doc_ids([drift_context], tables))

# 콜백이 모은 컨텍스트 전체로 넘긴 문서를 셉니다.
drift_ids = context_doc_ids(collector.contexts, tables)

print("콜백으로 모은 전체 문서 ID:", drift_ids)
print("M1 정답 근거 문서 ID:", question["gold_doc_ids"], "| 넘긴 정답 근거:", evidence_hit(drift_ids, question["gold_doc_ids"]), "편")

**확인 퀴즈**: 어떤 질문에서 Local이 Basic보다 정답 근거 문서를 적게 넘겼다면, Local이 더 나쁜 검색이라고 결론 내려도 될까요?

<details><summary>정답 보기</summary>

그렇게 결론 내릴 수 없습니다. 먼저 넘긴 문서 수(`docs_given`)를 봅니다. Local은 다음 청크가 토큰 한도를 넘으면 멈추므로 Basic보다 적은 문서를 넘길 수 있고, 정답 근거 문서를 넘긴 비율(포함률)은 문서를 많이 넘길수록 저절로 오릅니다. 질문 하나가 아니라 여러 질문의 결과 행을 모아 넘긴 문서 수와 `all_hit`를 함께 보고 판단합니다.

</details>

### 4-5. 문서 한 편 추가(갱신) 비용

새 안내문이 등록되면 색인을 갱신합니다. `build_index`는 문서를 인자로 받지 않고 설정의 `input/` 폴더를 읽으므로, 새 문서 파일을 `input/`에 넣고 `is_update_run=True`로 부릅니다. 그러면 GraphRAG가 다음 순서로 갱신합니다.

1. 이전 색인에 없던 **파일 이름**만 찾아, 그 문서에서 개체·관계를 뽑습니다.
2. 그 문서의 개체·관계만으로 커뮤니티를 나누고 커뮤니티 보고서를 씁니다.
3. 결과를 기존 테이블에 덧붙입니다. 기존 커뮤니티는 다시 나누지 않습니다.

갱신에는 한계가 둘 있습니다.

- 기존 커뮤니티 보고서는 다시 쓰지 않아, 갱신을 거듭하면 실제 그래프와 어긋납니다. 운영에서는 주기적으로 전체를 다시 색인합니다.
- 같은 이름으로 고친 문서나 지운 문서는 감지하지 못하므로, 전체를 다시 색인해야 반영됩니다.

아래 셀은 비교 세트에서 뺀 `extra.json`의 문서로 갱신하고, 다음 셀은 처음 색인과 LLM 호출 수를 비교합니다. 다시 측정하려면 프로젝트 폴더를 만드는 셀부터 다시 실행합니다.

In [ ]:
# 새 문서 한 편을 input 폴더에 복사합니다.
shutil.copy(data_dir / "본실습" / "docs" / f"{extra_id}.txt", input_dir)
# 갱신 호출만 세도록 0으로 되돌립니다.
usage.update(llm_calls=0, input_tokens=0, output_tokens=0)
start = time.time()
# build_index는 문서를 인자로 받지 않고 설정의 input 폴더를 읽습니다.
# is_update_run=True면 input 폴더에서 이전 색인에 없는 파일 이름만 골라 그 문서만 추출해 기존 색인에 합칩니다.
update_results = await api.build_index(config=config, method="standard", is_update_run=True)
update_seconds = time.time() - start
# litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
await asyncio.sleep(1)

# 단계(workflow)마다 성공과 실패를 출력합니다.
for result in update_results:
    if result.error is None:
        print("완료:", result.workflow)
    else:
        print("실패:", result.workflow, repr(result.error))

In [ ]:
update_usage = dict(usage, seconds=round(update_seconds, 1))
# 처음 색인과 한 표로 비교합니다.
index_costs = pd.DataFrame([index_usage, update_usage], index=["index", "update"])

display(index_costs)

### 4-6. 색인 그래프 시각화

GraphRAG가 만든 개체·관계를 pyvis로 그려 봅니다. pyvis는 NetworkX처럼 노드와 관계를 넣으면 마우스로 끌고 확대할 수 있는 HTML 그래프를 만들어 주는 라이브러리입니다.

- **색:** 같은 색은 같은 레벨 0 커뮤니티(가장 큰 묶음)입니다. 커뮤니티 보고서는 이 묶음마다 하나씩 쓰였습니다.
- **크기와 굵기:** 노드 크기는 이어진 관계 수(`degree`), 선 굵기는 관계 강도(`weight`)입니다.
- **마우스를 올리면** 개체·관계의 `description`이 보입니다. 확대하면 개체 이름이 보입니다.

그래프는 처음 색인 결과(`tables`)로 그리고, `output/graphrag_graph.html`로도 저장합니다.

In [ ]:
import html

from IPython.display import HTML
from pyvis.network import Network


def show_network(net, html_path):
    """pyvis 그래프 net을 html_path에 HTML 파일로 저장하고, 노트북 출력 칸에 띄웁니다.

    net: 노드와 관계를 넣은 pyvis Network
    html_path: 저장할 HTML 파일 경로. 이 파일을 브라우저로 열어도 같은 그래프가 나옵니다.
    """
    net.write_html(str(html_path))
    page = html_path.read_text(encoding="utf-8")

    # HTML 문서를 iframe 안에 넣어 노트북 출력 칸에 그대로 띄웁니다.
    display(HTML(f'<div><iframe srcdoc="{html.escape(page)}" width="100%" height="680" style="border:0"></iframe></div>'))


communities = tables["communities"]
# 레벨 0 커뮤니티마다 entity_ids(그 커뮤니티에 든 개체 id 목록)를 펼쳐 '개체 id: 커뮤니티 번호' dict를 만듭니다.
community_of = {}

for row in communities[communities["level"] == 0].itertuples():
    for entity_id in row.entity_ids:
        community_of[entity_id] = row.community

# height·width: 그림 크기. cdn_resources="in_line"은 그리기 스크립트를 HTML 안에 넣어 인터넷 없이도 열리게 합니다.
graphrag_net = Network(height="650px", width="100%", cdn_resources="in_line")

for row in tables["entities"].itertuples():
    # group이 같은 노드는 같은 색으로 그립니다. 커뮤니티에 들지 못한 개체는 -1입니다.
    # title은 마우스를 올리면 보이는 글, value는 노드 크기입니다.
    graphrag_net.add_node(row.title, label=row.title, title=row.description,
                          group=int(community_of.get(row.id, -1)), value=int(row.degree))

for row in tables["relationships"].itertuples():
    # 관계는 이름 없이 설명만 있으므로 설명을 마우스 글로 붙이고, 강도를 선 굵기로 씁니다.
    graphrag_net.add_edge(row.source, row.target, title=row.description, value=float(row.weight))

print("개체:", len(tables["entities"]), "개 | 관계:", len(tables["relationships"]), "개 | 레벨 0 커뮤니티:", len(communities[communities["level"] == 0]), "개")

show_network(graphrag_net, output_dir / "graphrag_graph.html")

## 5. 🖐️ 함께 따라하기: 장애 지식베이스에 GraphRAG 적용

### 5-1. 운영팀 질문과 작성 목표

`data/따라하기/`에는 **누리페이**(수업용 가상 핀테크 회사)의 장애 보고서 14편과 서비스 카탈로그·담당 조직·변경 기록 6편, 모두 20편이 있습니다. 운영팀의 골드 질문 8개는 단일 사실 2개, 멀티홉 4개, 전역 2개입니다. 예를 들면 다음과 같습니다.

- Q7(전역): "4월부터 9월까지 누리페이 장애 보고서에서 되풀이해 나타난 원인 유형은 무엇인가?"
- Q5(멀티홉): "5월 잔액 조회 오류 장애의 원인 서비스가 의존하는 캐시를 운영하는 팀의 온콜 담당자는 누구인가?" 정답 근거 문서는 서비스 이름과 캐시 이름으로 이어진 세 편(`INC-0518`, `CAT-03`, `ORG-02`)입니다.

**작성 목표**

1. 개체 유형을 서비스·구성요소·팀·인물·장애로 정해 색인합니다.
2. 질문마다 Jev가 질문 유형을 보고 네 검색 가운데 하나를 고르면, 그 검색으로 답합니다.
3. 결과를 `output/practice_results_lesson01.csv`로 저장합니다.

**힌트:** 본문의 색인·검색 코드를 참고하고 변수 이름에 `practice_`를 붙입니다. `make_settings`, `run_search`, `make_result_row`, `read_tables`, `context_doc_ids`는 다시 정의하지 않고 부릅니다. Jev 사용법은 검색 라우팅 단원과 같습니다.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **직접 설계:** 지시 블록에 적힌 이름·입력·반환값에 맞춰 코드 전체를 작성합니다.
- **부분 작성:** 코드의 `...` 자리만 채웁니다. 이미 적힌 경로·출력 코드는 그대로 실행합니다.
- `...`가 남은 채로 실행하면 오류가 나거나 빈 결과가 나옵니다. 작성 셀을 완성한 뒤 바로 아래 셀을 실행해 결과를 봅니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 5-2. 설정과 색인

**부분 작성 · 따라하기 프로젝트 설정**

**작성:**

1. `practice_project_dir`를 지우고(`shutil.rmtree(..., ignore_errors=True)`) `practice_input_dir`를 만든 뒤(`mkdir(parents=True)`), `data/따라하기/docs`의 txt 20편을 넣습니다.
2. `make_settings(practice_project_dir, [...])`로 `entity_types`가 `["서비스", "구성요소", "팀", "인물", "장애"]`인 설정 dict를 만들어 `practice_settings`에 담습니다.
3. `GraphRagConfig(**practice_settings)`로 설정 객체 `practice_config`를 만듭니다.

**반환값:** 출력이 `입력 문서: 20 편`이고 `entity_types`가 다섯 유형입니다.

In [ ]:
practice_project_dir = output_dir / "graphrag_따라하기"
practice_input_dir = practice_project_dir / "input"
# 1. practice_project_dir를 지우고 practice_input_dir를 만든 뒤, data/따라하기/docs의 txt 20편을 넣으세요.
...
# 2. make_settings로 entity_types가 ["서비스", "구성요소", "팀", "인물", "장애"]인 설정 dict를 practice_settings에 담으세요.
practice_settings = ...
# 3. GraphRagConfig로 설정 객체 practice_config를 만드세요.
practice_config = ...

print("입력 문서:", len(list(practice_input_dir.glob("*.txt"))), "편 | entity_types:", practice_config.extract_graph.entity_types, "| 원문 비율:", practice_config.local_search.text_unit_prop)

**부분 작성 · 따라하기 색인**

**작성:**

1. `usage`를 0으로 되돌린 뒤 `practice_config`로 `api.build_index`를 실행해 결과를 `practice_results`에 담습니다.
2. 걸린 시간을 `practice_seconds`에 담고, `await asyncio.sleep(1)`로 사용량 콜백을 기다린 뒤 `result.error`가 `None`이 아닌 단계만 `실패:` 줄로 출력합니다.

**반환값:** `practice_results`는 단계별 결과 목록이고, `실패:` 줄이 없어야 합니다.

In [ ]:
# 색인을 기다리기 어려우면 이 셀 대신 바로 아래 셀로 강사가 미리 만든 색인을 복사합니다. 둘 중 하나만 실행합니다.
# 1. usage를 0으로 되돌리고 practice_config로 build_index를 실행해, 결과를 practice_results에, 걸린 시간을 practice_seconds에 담으세요.
# 2. 1초 기다린 뒤 실패한 단계(result.error가 None이 아닌 결과)의 이름과 에러를 출력하세요.
...

print("따라하기 색인:", round(practice_seconds), "초 | LLM 호출:", usage["llm_calls"], "회 | 입력 토큰:", usage["input_tokens"], "| 출력 토큰:", usage["output_tokens"])

색인은 1~2분 걸리고 LLM을 약 80회 부르므로, 기다리기 어렵다면 위 색인 셀 대신 아래 셀로 강사가 미리 만든 색인(`data/따라하기/prebuilt/`)을 복사합니다. 두 셀 가운데 하나만 실행합니다.

In [ ]:
# 위 따라하기 색인 셀 대신 실행합니다. 색인 셀을 실행했다면 건너뜁니다.
# 강사가 같은 설정으로 만든 색인을 practice_project_dir/output에 복사합니다. 검색에 쓰는 practice_config는 설정 셀에서 이미 만들었습니다.
shutil.copytree(data_dir / "따라하기" / "prebuilt" / "graphrag", practice_project_dir / "output", dirs_exist_ok=True)
# 강사 색인의 호출 수·토큰·시간
practice_index_usage = read_json("따라하기/prebuilt/index_usage.json")["graphrag"]

print("복사한 색인 파일:", sorted(path.name for path in (practice_project_dir / "output").iterdir()))
print("강사 색인 시간:", round(practice_index_usage["seconds"]), "초 | LLM 호출:", practice_index_usage["llm_calls"], "회 | 입력 토큰:", practice_index_usage["input_tokens"], "| 출력 토큰:", practice_index_usage["output_tokens"])

### 5-3. 테이블 읽기와 검색 실행

**직접 설계 · 따라하기 테이블 읽기**

**작성:**

1. `read_tables`로 `practice_tables`를 만듭니다.
2. 테이블마다 이름과 행 수를 출력하고, 커뮤니티 보고서 레벨 목록을 출력합니다.

**반환값:** `documents` 테이블이 20행입니다.

In [ ]:
# 1. read_tables로 practice_tables를 만드세요.
# 2. 테이블마다 행 수를 출력하고, 커뮤니티 보고서 레벨 목록을 출력하세요.
...

실무에서는 질문 유형을 먼저 판단하고 그에 맞는 검색을 고릅니다. 단일 사실을 묻는데 Global을 쓰면 커뮤니티 보고서를 모두 읽느라 호출만 늘고, 멀티홉 질문에 Basic을 쓰면 질문과 표현이 다른 두 번째 문서가 빠지기 쉽습니다. 그래서 검색 라우팅 단원에서 쓴 Jev(`TypeSafeClassifier`)에 네 검색의 기준을 주고, 질문마다 하나를 고르게 합니다.

Jev가 고른 검색은 골드 질문의 유형과 다를 수 있습니다. 1위 확률이 낮고 2위와 비슷한 질문(예: 0.4 대 0.37)은 경계에 있어 실행마다 바뀔 수 있으므로, 결과 표에서 Jev 선택과 골드 유형을 함께 봅니다.

**부분 작성 · Jev 검색 방식 분류기**

**작성:** `practice_router`를 `TypeSafeClassifier`로 만듭니다. `model`은 환경변수 `TYPESAFE_MODEL`을 쓰고, 없으면 `"jev-1.13.0"`입니다. 네 검색의 기준(`practice_route_question`)은 제공합니다.

**반환값:** 출력에 Jev 모델 이름과 네 검색 방식(`basic`, `local`, `drift`, `global`)이 나옵니다.

In [ ]:
from langchain_typesafe import Choice, TypeSafeClassifier

# 1. practice_router를 TypeSafeClassifier로 만드세요. model은 환경변수 TYPESAFE_MODEL을 쓰고, 없으면 "jev-1.13.0"입니다.
practice_router = ...

# 질문마다 네 검색 가운데 하나를 고르는 기준입니다. 답을 얻기까지 몇 단계를 거치는지로 나눕니다.
practice_route_question = Choice(
    instructions="질문에 맞는 GraphRAG 검색 방식을 고르세요. 답을 얻기까지 거쳐야 하는 대상이 몇 단계인지, 자료 전체를 모아야 하는지 먼저 보세요.",
    criteria={
        "basic": "단일 사실: 한 문서에 적힌 수치·시간 같은 사실 하나를 묻고, 다른 대상으로 이어 갈 필요가 없는 질문",
        "local": "특정 개체 주변: 개체 하나를 정해 그 개체와 바로 이어진 정보를 두루 묻는 질문",
        "drift": "멀티홉: 한 문서의 답을 다른 문서의 대상으로 이어서 찾아야 하는, 연결을 두 단계 이상 따라가는 질문",
        "global": "전역: 특정 대상 하나가 아니라 자료 전체에 흩어진 내용을 모아 목록·유형·경향을 정리하는 질문",
    },
)

print("Jev 모델:", practice_router.model, "| 검색 방식:", list(practice_route_question.criteria))

**부분 작성 · Jev가 고른 검색으로 골드 질문 8개 검색**

**작성:**

1. `practice_router.invoke`에 `{"state": 질문 글, "questions": {"route": practice_route_question}}`를 넘겨 `route_decision`에 담습니다.
2. `route_decision.choices["route"].choice`를 `route`에 담습니다.
3. `route`로 `run_search`를 부릅니다. Global이 아니면 `context_doc_ids(contexts, practice_tables)`로 넘긴 문서 ID 목록을 `given_ids`에 담습니다.

**반환값:** `practice_rows`에 질문마다 한 행씩 8행이 쌓이고, 줄마다 질문 유형(`qtype`)과 Jev의 선택이 함께 출력됩니다. Global 줄의 `넘긴 문서 ID:`는 `None`입니다.

In [ ]:
practice_questions = read_json("따라하기/questions.json")
practice_answers = {}
practice_rows = []

for question in practice_questions:
    # 1. practice_router.invoke에 질문 글(state)과 {"route": practice_route_question}(questions)을 넘겨 route_decision에 담으세요.
    route_decision = ...
    # 2. route_decision.choices["route"]에서 Jev가 고른 검색 방식(choice)을 route에 담으세요.
    route = ...
    # Jev가 고른 값이 네 검색 가운데 하나인지 확인합니다.
    assert route in practice_route_question.criteria, route

    # 3. route로 run_search를 실행하세요. 설정·테이블은 practice_config·practice_tables를 넘깁니다.
    answer, contexts, seconds = ...
    practice_answers[question["id"]] = answer

    if route == "global":
        # Global은 원문을 넘기지 않아 문서 열을 빈칸으로 둡니다.
        given_ids = None
    else:
        # 4. context_doc_ids로 contexts의 청크를 문서 ID 목록으로 바꿔 given_ids에 담으세요.
        given_ids = ...

    row = make_result_row("graphrag_" + route, question, given_ids, usage, seconds)
    # Jev의 선택과 그 선택의 확률을 함께 남깁니다.
    row["route"] = route
    row["route_probability"] = round(route_decision.choices["route"].probabilities[route], 2)
    practice_rows.append(row)

    print(question["id"], question["qtype"], "| Jev 선택:", route, "| 넘긴 문서 ID:", given_ids)

Q7(전역) 답변입니다. Jev가 Global을 골랐다면 답 끝의 `[Data: Reports (번호)]`는 근거로 읽은 커뮤니티 보고서 번호입니다.

In [ ]:
display(Markdown(practice_answers["Q7"]))

**부분 작성 · 결과 표 저장**

**작성:** `practice_rows`로 DataFrame `practice_table`을 만들어 `output_dir / "practice_results_lesson01.csv"`로 저장합니다(`index=False`).

**반환값:** 표는 8행이고 열은 `make_result_row`가 만드는 11개(`method`부터 `seconds`까지)에 `route`, `route_probability`를 더한 13개입니다.

In [ ]:
# 1. practice_rows로 DataFrame practice_table을 만들어 output_dir / "practice_results_lesson01.csv"로 저장하세요.
practice_table = ...

display(practice_table)

## 6. 핵심 정리

- **색인:** 청크마다 개체·관계를 뽑고, Leiden 계층 묶음마다 커뮤니티 보고서를 씁니다. 색인 비용 대부분이 이 두 작업입니다.
- **스키마:** 사람은 개체 유형만 정하고, 관계는 설명 문장으로 저장됩니다. 표기가 다르면 같은 대상이 두 개체가 됩니다.
- **네 검색:** Basic은 가까운 청크를, Local은 질문 개체 주변의 관계·커뮤니티 보고서·원문을 넘깁니다. Global은 커뮤니티 보고서 전체를 map-reduce로 요약하고, DRIFT는 그중 가까운 몇 개로 첫 답을 만든 뒤 후속 질문마다 Local 검색을 합니다.
- **근거 비교:** 정답 근거 문서를 넘겼는지는 넘긴 문서 수와 함께 봅니다. DRIFT가 넘긴 문서는 콜백으로 모아야 빠짐없이 셉니다.
- **갱신:** `is_update_run=True`는 새 파일만 반영하고, 고친 파일은 감지하지 못합니다.
- **검색 고르기:** Jev가 질문 유형을 보고 검색을 고르게 하면, 단일 사실 질문에 Global처럼 호출이 많은 검색을 쓰지 않습니다.